# 06 — SCIN Dataset: Rotation Augmentation

Downloads SCIN images, creates rotation-augmented copies, and uploads
original + rotated images to our GCS bucket.

**What this does:**
1. Download SCIN images from public bucket → **verify** valid count
2. Apply random rotation (θ ∈ [10°, 175°], seed=42) → **verify** rotation
3. Upload to our bucket → **verify** upload count matches

All rotation logic is imported from `scripts/add_rotation.py` — no duplication.

**References:**
- SCIN dataset: https://github.com/google-research-datasets/scin
- Augmentation: https://pmc.ncbi.nlm.nih.gov/articles/PMC5977656/

In [ ]:
# Cell 1: Setup
import os, subprocess, sys

if not os.path.exists("/content/NST_Class"):
    subprocess.run(["git", "clone", "https://github.com/AayushBaniya2006/NST_Class.git"], cwd="/content")
else:
    subprocess.run(["git", "pull"], cwd="/content/NST_Class")
os.chdir("/content/NST_Class")

!pip install -q -r requirements.txt
sys.path.insert(0, "/content/NST_Class")

print("Setup complete!")

In [ ]:
# Cell 2: Authenticate with Google Cloud
from google.colab import auth
auth.authenticate_user()
print("Authenticated!")

In [ ]:
# Cell 3: Configuration
SOURCE_BUCKET = "dx-scin-public-data"
SOURCE_PREFIX = "dataset/images"
DEST_BUCKET = "skin-tone-project"
DEST_PREFIX = "scin_images"

LOCAL_IMAGE_DIR = "data/scin/images"
LOCAL_ROTATED_DIR = "data/scin/images_rotated"

ANGLE_MIN = 10.0
ANGLE_MAX = 175.0
SEED = 42

print(f"Source: gs://{SOURCE_BUCKET}/{SOURCE_PREFIX}")
print(f"Dest:   gs://{DEST_BUCKET}/{DEST_PREFIX}")
print(f"Rotation: angle=[{ANGLE_MIN}, {ANGLE_MAX}], seed={SEED}")

In [ ]:
# Cell 4: Download SCIN images (gcloud parallel) + verify
from scripts.add_rotation import verify_download

!mkdir -p {LOCAL_IMAGE_DIR}
!gcloud storage cp -r -n gs://{SOURCE_BUCKET}/{SOURCE_PREFIX}/ {LOCAL_IMAGE_DIR}/
# gcloud storage is Go-based — much faster than gsutil for bulk transfers
# -n = skip existing

dl_result = verify_download(LOCAL_IMAGE_DIR)
print(f"\nVerification: {dl_result['valid']} valid, {dl_result['corrupt']} corrupt out of {dl_result['total']}")
assert dl_result["valid"] > 0, "No valid images found!"

In [ ]:
# Cell 5: Apply rotation
from scripts.add_rotation import process_images

count_before = dl_result["valid"]
created, rotated_paths = process_images(
    LOCAL_IMAGE_DIR, LOCAL_ROTATED_DIR,
    angle_min=ANGLE_MIN, angle_max=ANGLE_MAX, seed=SEED,
)

print(f"\nOriginals:  {count_before}")
print(f"Created:    {created} rotated images")
assert created > 0, "No rotated images created!"

In [ ]:
# Cell 6: Verify rotation + visual spot-check
import matplotlib.pyplot as plt
import numpy as np
from PIL import Image
from pathlib import Path
from scripts.add_rotation import verify_rotation

rotation_result = verify_rotation(LOCAL_IMAGE_DIR, LOCAL_ROTATED_DIR)
print(f"Passed:        {rotation_result['passed']}")
print(f"Pairs checked: {rotation_result['checked']}")
print(f"Missing:       {len(rotation_result['missing'])}")
assert rotation_result["passed"], f"Rotation verification failed: {rotation_result}"

# Visual: show 3 original vs rotated pairs side-by-side
from src.data.prepare import _IMAGE_EXTENSIONS
_ext_set = {e.lower() for e in _IMAGE_EXTENSIONS}
originals = sorted(
    f for f in Path(LOCAL_IMAGE_DIR).rglob("*")
    if f.is_file() and f.suffix.lower() in _ext_set and "_rotated" not in f.stem
)[:3]

fig, axes = plt.subplots(len(originals), 2, figsize=(8, 4 * len(originals)))
if len(originals) == 1:
    axes = [axes]
for i, orig in enumerate(originals):
    rotated_path = Path(LOCAL_ROTATED_DIR) / f"{orig.stem}_rotated{orig.suffix}"
    axes[i][0].imshow(Image.open(orig))
    axes[i][0].set_title(f"Original: {orig.name}")
    axes[i][0].axis("off")
    if rotated_path.exists():
        axes[i][1].imshow(Image.open(rotated_path))
        axes[i][1].set_title(f"Rotated: {rotated_path.name}")
    axes[i][1].axis("off")
plt.tight_layout()
plt.show()

In [ ]:
# Cell 7: Upload rotated images to bucket + verify
# Originals already uploaded by notebook 05 — only upload rotated
import os
from scripts.add_rotation import verify_upload

ROTATED_PREFIX = f"{DEST_PREFIX}/rotated"

# Increase parallelism for faster uploads (defaults: 4 threads, 1 process)
os.environ["CLOUDSDK_STORAGE_THREAD_COUNT"] = "16"
os.environ["CLOUDSDK_STORAGE_PROCESS_COUNT"] = "4"

# -n = skip already-uploaded files on re-run
!gcloud storage cp -r -n {LOCAL_ROTATED_DIR}/ gs://{DEST_BUCKET}/{ROTATED_PREFIX}/

# Verify only the rotated prefix (not the whole scin_images/ which has noised too)
upload_result = verify_upload(DEST_BUCKET, ROTATED_PREFIX, created)
print(f"\nUpload verification: {upload_result['actual_count']}/{upload_result['expected_count']} files")
assert upload_result["passed"], f"Upload verification failed: {upload_result}"

In [ ]:
# Cell 8: Summary
print("=" * 60)
print("ROTATION AUGMENTATION PIPELINE SUMMARY")
print("=" * 60)
print(f"Download:  {dl_result['valid']} valid images ({dl_result['corrupt']} corrupt)")
print(f"Rotation:  {created} rotated images (angle=[{ANGLE_MIN}, {ANGLE_MAX}], seed={SEED})")
print(f"Upload:    {upload_result['actual_count']} files in gs://{DEST_BUCKET}/{DEST_PREFIX}")
print(f"")
print(f"Verify download: PASS ({dl_result['valid']} valid)")
print(f"Verify rotation: {'PASS' if rotation_result['passed'] else 'FAIL'}")
print(f"  checked={rotation_result['checked']} pairs, missing={len(rotation_result['missing'])}")
print(f"Verify upload:   {'PASS' if upload_result['passed'] else 'FAIL'}")
print(f"  {upload_result['actual_count']}/{upload_result['expected_count']} files")
print("=" * 60)
print("All checks passed!" if (rotation_result['passed'] and upload_result['passed']) else "SOME CHECKS FAILED")